In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key=os.getenv("sample_qroq_api_key")


In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader('./attention.pdf')
docs=loader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukas

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
final_documents=splitter.split_documents(docs)
final_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to'),
 Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='reproduce the tables 

In [21]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [22]:
from langchain_chroma import Chroma

vector_db=Chroma.from_documents(embedding=embeddings,documents=final_documents)
vector_db

In [23]:
retriever=vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={})

In [34]:
from langchain_groq import ChatGroq

import httpx


custom_http_client = httpx.Client(verify=False)
llm=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",groq_api_key=groq_key,http_client=custom_http_client)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39915e450>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34dd99690>)

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
        ("system",system_prompt),
        ("human","{input}")
    ]
)



prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [26]:
# # from langchain_core.prompts import ChatPromptTemplate, SystemMessage, HumanMessage

# # Define the prompt template with input variables
# prompt = ChatPromptTemplate(
#     input_variables=["context", "input"],
#     messages=[
#         SystemMessage(content=(
#             "You are an assistant for question-answering tasks. "
#             "Use the following pieces of retrieved context to answer the question. "
#             "If you don't know the answer, say that you don't know. "
#             "Use three sentences maximum and keep the answer concise.\n\n{context}"
#         )),
#         HumanMessage(content="{input}")
#     ]
# )

# # Example values for the variables
# context_text = "The Eiffel Tower is located in Paris and was completed in 1889."
# question_text = "Where is the Eiffel Tower located?"

# # Format the prompt with those values
# formatted_prompt = prompt.format_prompt(context=context_text, input=question_text)

# # Print the resulting messages ready for the model
# for message in formatted_prompt.messages:
#     print(f"{message.type}: {message.content}")


# formatted_prompt


In [27]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough,RunnableMap

# method1

In [36]:
chain1={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt|llm
chain1

{
  context: RunnableLambda(lambda x: retriever.get_relevant_documents(x['input'])),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39915e450>, model_name='meta-llama/llama-4-scout-17b-16

In [29]:
retriever.get_relevant_documents("what is encoder")

[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='en

In [30]:
## will not fill values
#Because prompt.invoke() doesn’t run Runnables — it just substitutes values.


prompt.invoke({
    "context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nRunnableLambda(...)", additional_kwargs={}, response_metadata={}), HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={})])

In [31]:
prompt.invoke({
    "context": retriever.get_relevant_documents("what is encoder"),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'), Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00',

In [32]:
chain_x={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'), Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00',

In [37]:
chain1.invoke({"input": "what is encoder"})

AIMessage(content='The encoder is not explicitly defined in the provided context, but based on the mentions of "encoder stack" and "output of the encoder," it appears to be a component of a neural network or a similar model. The context seems to be discussing attention mechanisms in relation to the encoder. I can\'t provide a more specific definition without additional information.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 892, 'total_tokens': 961, 'completion_time': 0.1662613, 'prompt_time': 0.029177928, 'queue_time': 0.049041542, 'total_time': 0.195439228}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--f7392227-c092-4456-8640-75a5fd8824c1-0', usage_metadata={'input_tokens': 892, 'output_tokens': 69, 'total_tokens': 961})

# method 2

In [38]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


context_retrieval_chain = (
    RunnableLambda(lambda x: x["input"]) |
    retriever |
    RunnableLambda(format_docs)
)

In [39]:
chain2={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt|llm
chain2

{
  context: RunnableLambda(...)
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completi

In [40]:
context_retrieval_chain.invoke({"input": "what is encoder"})

'encoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual'

In [41]:
prompt.invoke({
    "context": context_retrieval_chain.invoke({"input": "what is encoder"}),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual", additional_kwargs={}, response_metadata={}), HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={})])

In [42]:
chain_x={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual", additional_kwargs={}, response_metadata={}), HumanMessage(content="{'input': 'what is encoder'}", additional_kwargs={}, response_metadata={})])

In [43]:
chain2.invoke({"input": "what is encoder"})

AIMessage(content='The encoder is a component that processes input sequences. It employs a stack with residual connections. The output of the encoder stack is used for further processing.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 109, 'total_tokens': 140, 'completion_time': 0.071487854, 'prompt_time': 0.002790908, 'queue_time': 0.051351122, 'total_time': 0.074278762}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--302ee8a6-d827-4819-a27e-6bf3a3886093-0', usage_metadata={'input_tokens': 109, 'output_tokens': 31, 'total_tokens': 140})

# method3

In [44]:
chain3=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])) 
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt|llm
chain3

{
  context: RunnableLambda(lambda x: retriever.get_relevant_documents(x['input']))
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39915e450>, mo

In [45]:
retriever.get_relevant_documents("what is encoder")

[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='en

In [46]:
RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
).invoke({"input": "what is encoder"})

{'context': 'encoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual',
 'input': {'input': 'what is encoder'}}

In [47]:
# error
prompt.invoke(RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
).invoke({"input": "what is encoder"}))

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual", additional_kwargs={}, response_metadata={}), HumanMessage(content="{'input': 'what is encoder'}", additional_kwargs={}, response_metadata={})])

In [48]:
chain_x=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual", additional_kwargs={}, response_metadata={}), HumanMessage(content="{'input': 'what is encoder'}", additional_kwargs={}, response_metadata={})])

In [49]:
chain3.invoke({"input": "what is encoder"})

AIMessage(content="The encoder is a component that processes input sequences. It employs a stack with residual connections. I don't know more details about this specific encoder.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 109, 'total_tokens': 138, 'completion_time': 0.065529075, 'prompt_time': 0.002933346, 'queue_time': 0.053432073, 'total_time': 0.068462421}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--85a7b533-9cf0-4802-be52-29cb09e18bbe-0', usage_metadata={'input_tokens': 109, 'output_tokens': 29, 'total_tokens': 138})

# method 4

In [50]:
retriever1=vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriever1

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={'k': 3})

In [51]:
#LangChain internally wraps retriever1 so that when it receives {"input": "your query"}, 
# it automatically uses "input" as the query string — if no other fields are specified.


chain4={"context": retriever1,"input":RunnablePassthrough()}|prompt|llm
chain4

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={'k': 3}),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.comple

In [52]:
retriever1.invoke("what is encoder")

[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='en

In [53]:
prompt.invoke({"context":retriever1.invoke("what is encoder"),"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'), Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00',

In [54]:
chain_x = {
    "context": retriever1,             # already a Runnable
    "input": RunnablePassthrough()     # passes the question as-is
}| prompt
chain_x.invoke("what is encoder")

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'), Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00',

In [55]:
#this will raise error because 
# RunnableMap gives the entire input dict (i.e., {"input": "what is encoder"}) to both retriever1 and RunnablePassthrough().


# chain_x = {
#     "context": retriever1,             # already a Runnable
#     "input": RunnablePassthrough()     # passes the question as-is
# }| prompt
# chain_x.invoke({"input":"what is encoder"})

In [56]:
chain4.invoke("where is softmax used?")

AIMessage(content='Softmax is commonly used in machine learning models, particularly in the output layer of classification models. It is used to normalize the output of a model to ensure that it forms a valid probability distribution over multiple classes. This allows the model to predict probabilities for each class.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 693, 'total_tokens': 747, 'completion_time': 0.123702969, 'prompt_time': 0.024022002, 'queue_time': 0.297627438, 'total_time': 0.147724971}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--66d9cf99-722a-47f3-9bf2-25b97c880047-0', usage_metadata={'input_tokens': 693, 'output_tokens': 54, 'total_tokens': 747})

# method 5

In [57]:
retriever1.invoke("what is encoder")

[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='en

In [58]:
chain5={"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt|llm
chain5

{
  context: RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.ch

In [59]:
retriever1.invoke("what is encoder")

[Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='en

In [60]:
format_docs(retriever1.invoke("what is encoder"))

'encoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual'

In [61]:
chain_x = {"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt
chain_x.invoke({"input":"what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual", additional_kwargs={}, response_metadata={}), HumanMessage(content="{'input': 'what is encoder'}", additional_kwargs={}, response_metadata={})])

In [62]:
chain5.invoke({"input":"what is encoder"})

AIMessage(content='The encoder is a component that processes input data. It is typically a stack of layers that transforms the input into a higher-level representation. The output of the encoder is often used as input to other components, such as the decoder or attention mechanisms.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 91, 'total_tokens': 141, 'completion_time': 0.122600004, 'prompt_time': 0.002364035, 'queue_time': 0.053101375, 'total_time': 0.124964039}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--2c2c978a-b7c0-484a-a98d-db25ce9f92b6-0', usage_metadata={'input_tokens': 91, 'output_tokens': 50, 'total_tokens': 141})

# method6

In [63]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

stuffed_doc=create_stuff_documents_chain(prompt=prompt,llm=llm)
chain6=create_retrieval_chain(retriever,stuffed_doc)
print(stuffed_doc)
chain6

bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39915e45

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x393f58810>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answe

In [64]:
chain6.invoke({"input":"what is encoder"})

{'input': 'what is encoder',
 'context': [Document(id='07e17fd8-7c69-40c8-96c3-5e16a5e2f993', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
  Document(id='cf451275-d36f-471b-a212-8908bebc0727', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 

Great question — you're now using a more **abstracted and higher-level** way of building LangChain chains using:

* `create_stuff_documents_chain(...)`
* `create_retrieval_chain(...)`

These functions **hide a lot of internal wiring**, so understanding how the chain maps `"input"` and `"output"` is key to using it correctly.

---

## 🔍 Your Code Summary

```python
stuffed_doc = create_stuff_documents_chain(prompt=prompt, llm=llm)
chain6 = create_retrieval_chain(retriever, stuffed_doc)

response = chain6.invoke({"input": "where is softmax used?"})
```

---

## ✅ What Each Step Does

### 1. `create_stuff_documents_chain(prompt, llm)`

This creates a **Chain** where:

* The `retrieved documents` are **stuffed** (i.e., concatenated) into a prompt.
* The prompt is passed to the `llm`.

This chain expects:

```python
{"context": List[Document], "input": str}
```

LangChain will then:

* Format the prompt using both `{context}` and `{input}`.
* Pass it to the LLM.
* Return the LLM’s response.

📦 Output:

```python
{"output": "<llm_response>"}
```

---

### 2. `create_retrieval_chain(retriever, stuffed_doc)`

This wraps your `stuffed_doc` chain with a retriever.

It does the following steps internally:

1. Takes your input: `{"input": "where is softmax used?"}`
2. Runs `retriever.invoke("where is softmax used?")` to get documents.
3. Passes to `stuffed_doc`:

   ```python
   {
       "input": "where is softmax used?",
       "context": [Document1, Document2, ...]
   }
   ```
4. Receives the LLM output from `stuffed_doc`.
5. Returns a dict like:

   ```python
   {
       "input": "where is softmax used?",
       "context": [Document1, Document2, ...],
       "answer": "<llm response>"
   }
   ```

---

## 🎯 Final Mapping

Here’s a clear breakdown of what's happening in `chain6.invoke(...)`:

| Step                       | Input / Output                                             | Description                         |
| -------------------------- | ---------------------------------------------------------- | ----------------------------------- |
| `invoke({"input": "..."})` | Input dict with question                                   | Your initial input                  |
| `retriever`                | `"where is softmax used?"` → `[Document1, Document2, ...]` | Retrieves relevant context          |
| `stuffed_doc`              | `{input: ..., context: [...]}`                             | Builds prompt and sends to LLM      |
| `llm`                      | Generates final answer                                     | From stuffed prompt                 |
| `create_retrieval_chain`   | Returns full dict                                          | Includes input, context, and answer |

---

## 📥 Input Format

You must provide:

```python
{"input": "<your question>"}
```

## 📤 Output Format

You get back:

```python
{
    "input": "<your question>",
    "context": [<retrieved documents>],
    "answer": "<llm-generated answer>"
}
```

---

## 🔁 Example

```python
response = chain6.invoke({"input": "where is softmax used?"})
```

Sample output:

```python
{
    'input': 'where is softmax used?',
    'context': [Document(page_content="Softmax is used in neural networks..."), ...],
    'answer': 'Softmax is commonly used in the final layer of classification neural networks to convert logits into probabilities.'
}
```

You can access the result like:

```python
print(response["answer"])
```

---

## ✅ Summary

* `create_retrieval_chain()` automatically handles input mapping.
* It expects a key `"input"` and returns keys `"input"`, `"context"`, and `"answer"`.
* Internally, it maps `"input"` → retriever → `"context"` → prompt → LLM → `"answer"`.

---

Let me know if you want to:

* Customize the returned output (e.g. exclude `"context"`),
* Switch to a `map_reduce` or `refine` document chain instead of `stuff`,
* Use async for batching.


In [65]:
chain6.invoke({"input":"where is softmax used?"})

{'input': 'where is softmax used?',
 'context': [Document(id='50b3e5ed-c727-4686-9313-8d5913071073', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='of the softmax which correspond to illegal connections. See Figure 2.'),
  Document(id='59d28510-02ca-4cfa-a8aa-6c44a52687fc', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'sou

## adding history basic example

In [68]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(llm,get_session_history)
with_message_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x3a8a9e980>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [69]:
config1={"configurable":{"session_id":"chat1"}}

In [70]:
with_message_history.invoke(
    [
        HumanMessage(content="Hi,my name is anurag"),
        HumanMessage(content="what is my name?"),
    ],
    config=config1)

AIMessage(content='Your name is Anurag! Nice to meet you!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 28, 'total_tokens': 41, 'completion_time': 0.029618899, 'prompt_time': 0.000730675, 'queue_time': 0.054218805, 'total_time': 0.030349574}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_5d3e4e58e1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--3bb50837-9201-492a-87a9-d488dcaa14d7-0', usage_metadata={'input_tokens': 28, 'output_tokens': 13, 'total_tokens': 41})

In [71]:
config2={"configurable":{"session_id":"chat2"}}

In [72]:
with_message_history.invoke(
    [
        HumanMessage(content="what is my name?"),
    ],
    config=config2)

AIMessage(content="I'm Meta AI. Think of me like an assistant who's here to help you learn, plan, and create. How can I assist you?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 15, 'total_tokens': 44, 'completion_time': 0.070456295, 'prompt_time': 6.9629e-05, 'queue_time': 0.053188831, 'total_time': 0.070525924}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--5afa1ed8-b3a3-42f5-9ee3-aade7b2e3cd3-0', usage_metadata={'input_tokens': 15, 'output_tokens': 29, 'total_tokens': 44})

## adding history above implemntation

## method1

In [73]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)



prompt = ChatPromptTemplate.from_messages([
        ("system",system_prompt),
        ("human","{input}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [74]:
model1=prompt|llm
model1

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x399149cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39915e450>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34dd99690>)

In [75]:
store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


In [76]:
with_message_history1=RunnableWithMessageHistory(model1,get_session_history,input_messages_key="input")
with_message_history1

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  input: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x3a8a9f4c0>, input_messages_key='input', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [77]:
config={"configurable":{"session_id":"chat1"}}

In [78]:
#here context become history
with_message_history1.invoke({"context":"my name is anurag","input":"what is my name?"},config=config)

AIMessage(content='Your name is Anurag. I was given this information earlier.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 86, 'total_tokens': 101, 'completion_time': 0.038893765, 'prompt_time': 0.002337581, 'queue_time': 0.054434249, 'total_time': 0.041231346}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_5d3e4e58e1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--6097a12c-87cd-4275-b4d2-3c91a23128ad-0', usage_metadata={'input_tokens': 86, 'output_tokens': 15, 'total_tokens': 101})

In [79]:
config={"configurable":{"session_id":"chat2"}}

In [80]:
# here retriver1  by default pass input query to it and pass context as history

# with_message_history1.invoke({"context":retriever1,"input":"where softmax function used?"},config=config)
with_message_history1.invoke({"context":retriever1|RunnableLambda(format_docs),"input":"where softmax function used?"},config=config)

AIMessage(content="The softmax function is commonly used in machine learning models, particularly in the output layer of classification models, such as logistic regression, neural networks, and more. It's used to normalize the output of a model to ensure it forms a valid probability distribution over multiple classes. I don't know the specific context or application you're referring to.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 134, 'total_tokens': 199, 'completion_time': 0.147139496, 'prompt_time': 0.003541133, 'queue_time': 0.052166726, 'total_time': 0.150680629}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--736448ae-69b7-44f9-862b-bfe157b5ac7d-0', usage_metadata={'input_tokens': 134, 'output_tokens': 65, 'total_tokens': 199})

In [81]:
store["chat2"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='where softmax function used?', additional_kwargs={}, response_metadata={}), AIMessage(content="The softmax function is commonly used in machine learning models, particularly in the output layer of classification models, such as logistic regression, neural networks, and more. It's used to normalize the output of a model to ensure it forms a valid probability distribution over multiple classes. I don't know the specific context or application you're referring to.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 134, 'total_tokens': 199, 'completion_time': 0.147139496, 'prompt_time': 0.003541133, 'queue_time': 0.052166726, 'total_time': 0.150680629}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--736448ae-69b7-44f9-862b-bfe157b5ac7d-0', usage_metada

## importace of history_messages_key="history"

Got it! You're referring to the detailed parameters for **`RunnableWithMessageHistory`** in LangChain, especially the ones used to configure how history is passed in/out of the runnable. Let's break these down **with explanations and examples** so you can understand when and why to use each.

---

# RunnableWithMessageHistory: Parameters Explained

---

### 1. **`get_session_history`**

* **What:**
  A function that, given a session ID (string), returns an instance of `BaseChatMessageHistory` (or subclass) for that session.

* **Why:**
  This allows the runnable to **load and maintain the chat history for a specific user/session**. Each session has its own conversation history.

* **When to use:**
  When you want your runnable to support **multiple users or sessions**, each with their own stored message history.

* **Example:**

```python
from langchain.schema import BaseChatMessageHistory

def get_history(session_id: str) -> BaseChatMessageHistory:
    # Return the chat message history instance for this session_id
    return MyCustomChatMessageHistory.load(session_id)

runnable = RunnableWithMessageHistory(
    base_runnable=my_base_runnable,
    get_session_history=get_history,
    ...
)
```

---

### 2. **`input_messages_key`**

* **What:**
  If your base runnable takes a **dictionary** as input, this parameter tells the `RunnableWithMessageHistory` **which key in the input dict holds the new input messages** (list of messages or string).

* **Why:**
  To correctly extract the "current input" messages from a potentially larger input dict.

* **When to use:**
  When the input to your runnable is not a plain string but a dict containing multiple fields, including messages.

* **Example:**

Suppose your runnable expects input like:

```python
input_dict = {
    "user_id": "1234",
    "current_messages": [...],  # new input messages here
    "other_data": "foo"
}
```

You set:

```python
input_messages_key = "current_messages"
```

---

### 3. **`output_messages_key`**

* **What:**
  If your base runnable returns a **dictionary** as output, this tells which key in the output dict contains the output messages.

* **Why:**
  To correctly extract the generated messages from the runnable's output dict.

* **When to use:**
  When your runnable returns outputs in a dictionary, not just a string or list of messages.

* **Example:**

Runnable output might be:

```python
{
  "response_text": "Hello!",
  "messages": [...],  # output messages here
  "metadata": {...}
}
```

You set:

```python
output_messages_key = "messages"
```

---

### 4. **`history_messages_key`**

* **What:**
  If your base runnable **expects a separate key for the historical messages in the input dict**, specify this key.

* **Why:**
  To tell the runnable where to find past conversation history in the input.

* **When to use:**
  When your runnable’s input dict contains both new input messages *and* a separate key holding historical messages.

* **Example:**

Input dict might look like:

```python
{
  "new_messages": [...],
  "past_history": [...]  # historical messages here
}
```

You set:

```python
input_messages_key = "new_messages"
history_messages_key = "past_history"
```

---

### 5. **`history_factory_config`**

* **What:**
  Configuration dict for fields to pass to the chat history factory (`BaseChatMessageHistory` constructor). Controls things like max tokens, storage options, etc.

* **Why:**
  To customize how the chat history is created or loaded.

* **When to use:**
  When your chat history requires configuration, e.g., setting max length or special storage.

* **Example:**

```python
history_factory_config = {
    "max_length": 100,
    "storage_path": "/tmp/chat_histories"
}
```

---

# How These Work Together: Full Example

Let's say you have a base runnable that accepts a dict input like this:

```python
{
  "new_messages": [...],         # new input messages
  "past_messages": [...]         # history messages
}
```

and returns output like:

```python
{
  "result_text": "...",
  "generated_messages": [...]
}
```

You want to support per-session chat histories stored somewhere.

```python
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    # Load from some storage or DB by session_id
    return MyChatMessageHistory.load(session_id)

runnable_with_history = RunnableWithMessageHistory(
    base_runnable=my_base_runnable,
    get_session_history=get_session_history,
    input_messages_key="new_messages",
    history_messages_key="past_messages",
    output_messages_key="generated_messages",
    history_factory_config={
        "max_length": 50
    }
)
```

Now, when you run:

```python
response = runnable_with_history.invoke(
    {
      "session_id": "user123",
      "new_messages": [UserMessage(content="Hello!")],
      "past_messages": []
    }
)
```

* The runnable uses `get_session_history("user123")` to load the conversation history.
* It extracts new messages from `"new_messages"` key.
* It extracts past messages from `"past_messages"` key.
* It gets output messages from `"generated_messages"` key.

---

# Summary Table of Parameters

| Parameter                | Purpose                                       | Example Value                                   |
| ------------------------ | --------------------------------------------- | ----------------------------------------------- |
| `get_session_history`    | Function: session\_id → Chat history instance | `lambda session_id: MyHistory.load(session_id)` |
| `input_messages_key`     | Key for new input messages in input dict      | `"new_messages"`                                |
| `output_messages_key`    | Key for messages in output dict               | `"generated_messages"`                          |
| `history_messages_key`   | Key for history messages in input dict        | `"past_messages"`                               |
| `history_factory_config` | Config dict for history creation              | `{"max_length": 100}`                           |

---

If you'd like, I can help write a **minimal working example** or answer questions on how to integrate this in your specific LangChain use case. Would you want that?


In [82]:
#method1

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain1={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt|llm

runnable = RunnableWithMessageHistory(
    chain1,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)


Answer 1: The encoder is not explicitly defined in the provided context, but based on the mentions of "the output of the encoder," it appears to be a component that generates output used by other parts of a system, possibly in the context of machine learning or information processing. I don't know the specific details or definition of "encoder" in this context.
Answer 2: The decoder is composed of a stack of N = 6 identical layers, and it has self-attention layers that allow each position in the decoder to attend to other positions. I don't know more specific details about the decoder's function or definition beyond this.
Answer 3: The Applications of Attention in the model are described under section 3.2.3 of the document. The provided context mentions "3.2.3 Applications of Attention in our Model" on pages 5 of the document. I don't know the specific details of the applications beyond this.


In [83]:
store

{'da9ea904-4d54-4245-8f09-83cba2bb22cd': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is encoder?', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is not explicitly defined in the provided context, but based on the mentions of "the output of the encoder," it appears to be a component that generates output used by other parts of a system, possibly in the context of machine learning or information processing. I don\'t know the specific details or definition of "encoder" in this context.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 892, 'total_tokens': 961, 'completion_time': 0.16736156, 'prompt_time': 0.031951899, 'queue_time': 0.052193401, 'total_time': 0.199313459}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--d1f280b2-2e20-4621-ae16-f1cad1990c0e-0', 

In [84]:
#method2

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


context_retrieval_chain = (
    RunnableLambda(lambda x: x["input"]) |
    retriever |
    RunnableLambda(format_docs)
)
chain2={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt|llm
chain2

runnable = RunnableWithMessageHistory(
    chain2,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a part of a model, but I don't know what specific type of model or its function. It seems to have an output used for memory keys and values. I don't know more details about the encoder.
Answer 2: The decoder is composed of a stack of 6 identical layers. It has self-attention layers that allow each position in the decoder to attend to all positions in the decoder. The decoder seems to be part of a model, possibly a transformer-based model.
Answer 3: The applications of attention in the model are described in section 3.2.3, specifically "Applications of Attention in our Model". I don't know the specific details of these applications beyond their heading. More context is needed to provide further information.


In [85]:
#method3

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain3=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])) 
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt|llm

runnable = RunnableWithMessageHistory(
    chain3,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a part of a model that generates output based on input. In this context, the encoder's output is used to derive memory keys and values. I don't have more specific information about what the encoder is.
Answer 2: The decoder is composed of a stack of 6 identical layers. It allows each position in the decoder to attend to all positions in the decoder. The decoder generates output based on the output of the encoder.
Answer 3: The applications of attention in the model are described in section 3.2.3 of the context, specifically under "Applications of Attention in our Model". However, I don't have the specific information about what those applications are. The context only provides a header for the section, but not the actual content.


In [87]:
# #method4

# #this will fail always

# import uuid

# system_prompt = (
#     "You are an assistant for question-answering tasks. "
#     "Use the following pieces of retrieved context to answer "
#     "the question. If you don't know the answer, say that you "
#     "don't know. Use three sentences maximum and keep the "
#     "answer concise."
#     "\n\n"
#     "{context}"
# )

# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}")
# ])



# # 4. Create dummy documents to simulate a retriever
# from langchain.docstore.document import Document

# docs = [
#     Document(page_content="LangChain is a framework for building LLM-powered applications."),
#     Document(page_content="FAISS is a library for efficient similarity search."),
#     Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
# ]



# store={}
# def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
#     if session_id not in store:
#         store[session_id] = ChatMessageHistory()

    
#     return store[session_id]

# # 7. Combine all using RunnableWithMessageHistory
# #    Use retriever to get context, inject into prompt, then pass to model
# # chain = (
# #     {
# #         "context": lambda x: retriever.get_relevant_documents(x["input"]),
# #         "input": lambda x: x["input"]
# #     }
# #     | prompt
# #     | llm
# # )

# retriever1=vector_db.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k":3}
# )



# chain4={"context": retriever1,"input":RunnablePassthrough()}|prompt|llm
# chain4

# runnable = RunnableWithMessageHistory(
#     chain4,
#     # lambda session_id: get_session_history1(session_id),
#     get_session_history,
#     input_messages_key="input",
#     # history_messages_key="history"
# )

# # 8. Use it in a session
# session_id = str(uuid.uuid4())  # Generate unique session ID

# # Ask first question
# response1 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 1:", response1.content)

# # Ask second question (with context retained)
# response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 2:", response2.content)

# # Ask third question (could be unrelated or test memory/context)
# response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 3:", response3.content)



In [88]:
#method5

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain5={"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt|llm
chain5

runnable = RunnableWithMessageHistory(
    chain5,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a component that processes input data. Its output is used to generate memory keys and values. I don't have more specific information about the encoder's definition or function in this context.
Answer 2: The decoder is a component composed of a stack of 6 identical layers. It uses self-attention layers to allow each position in the decoder to attend to all positions in the encoder. The decoder generates output based on the memory keys and values from the encoder.
Answer 3: The applications of attention in the model are described in section 3.2.3. However, I don't have more specific information about the applications of attention. The context only mentions that there is a section about applications of attention in the model.


In [89]:
#method6

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

stuffed_doc=create_stuff_documents_chain(prompt=prompt,llm=llm)
chain6=create_retrieval_chain(retriever,stuffed_doc)

runnable = RunnableWithMessageHistory(
    chain6,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    output_messages_key="answer"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1['answer'])

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2['answer'])

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3['answer'])



Answer 1: The encoder is a part of a model, likely a type of neural network, but I don't have more specific information about its context or function. In general, an encoder is a component that transforms input data into a different representation. Without more context, I can't provide a more detailed explanation.
Answer 2: The decoder is composed of a stack of 6 identical layers. It uses self-attention layers to allow each position in the decoder to attend to all positions. This enables the decoder to generate output based on the input it receives.
Answer 3: The provided context does not specify the applications of attention. However, it appears that attention mechanisms have various applications in a model, as mentioned in section 3.2.3. The exact applications are not described in the given context.
